In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import shutil
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models.video as models
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader
import uuid
import torchvision.transforms.functional as F




In [ ]:
!unzip "/content/drive/MyDrive/proc_labels.zip" -d "/content/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/content/processed_labels/PASS/PASS_813_2019_10_01___Middlesbrough___Preston_North_End.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_067_2019_10_01___Brentford___Bristol_City.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_1114_2019_10_01___Brentford___Bristol_City.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_1331.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_1208_2019_10_01___Brentford___Bristol_City.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_578_2019_10_01___Hull_City___Sheffield_Wednesday.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_210_2019_10_01___Middlesbrough___Preston_North_End.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_1319_2019_10_01___Hull_City___Sheffield_Wednesday.mp4  
  inflating: /content/content/processed_labels/PASS/PASS_1583.mp4  
  inflating: /content/content/processed_lab

In [ ]:
DATA_DIR = "/content/content/processed_labels"
NUM_FRAMES = 32
IMAGE_SIZE = 112
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 5e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

🚀 Using device: cuda


In [ ]:
def merge_rare_classes(base_path):
    target_folder = "SHOT"
    target_path = os.path.join(base_path, target_folder)
    os.makedirs(target_path, exist_ok=True)

    for folder in ["GOAL", "FREE_KICK"]:
        source_path = os.path.join(base_path, folder)
        if os.path.exists(source_path):
            for file in os.listdir(source_path):
                if file.endswith('.mp4'):
                    shutil.move(os.path.join(source_path, file), os.path.join(target_path, file))
            os.rmdir(source_path)
            print(f"✅ Merged {folder} into {target_folder}")

print("Checking folders...")
merge_rare_classes(DATA_DIR)

# Dynamically get the final list of classes (ignoring hidden system folders)
CLASSES = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith('.')])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f"🎯 Final Classes ({len(CLASSES)}): {CLASSES}")

Checking folders...
✅ Merged GOAL into SHOT
✅ Merged FREE_KICK into SHOT
🎯 Final Classes (9): ['BALL_PLAYER_BLOCK', 'CROSS', 'HEADER', 'HIGH_PASS', 'OUT', 'PASS', 'PLAYER_SUCCESSFUL_TACKLE', 'SHOT', 'THROW_IN']


In [ ]:
# ==========================================
# 3. SPLIT BY MATCH (NO LEAKAGE + FALLBACK!)
# ==========================================
import uuid
import os
import random

video_paths = []
labels = []
match_names = []

for cls_name in CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls_name)
    for vid_file in os.listdir(cls_dir):
        if not vid_file.endswith('.mp4'): continue

        video_paths.append(os.path.join(cls_dir, vid_file))
        labels.append(CLASS_TO_IDX[cls_name])

        # Extract match name if it exists
        if "___" in vid_file:
            match_name = vid_file.split("___", 1)[1].replace(".mp4", "")
        else:
            # FALLBACK: Give it a unique ID so it gets split normally
            match_name = f"STANDALONE_{uuid.uuid4().hex}"

        match_names.append(match_name)

unique_matches = list(set(match_names))
total_videos = len(video_paths)

print("⚖️ Balancing the train/val split based on video counts...")

# THE FIX: Keep shuffling until the validation set gets ~20% of the actual videos!
while True:
    random.shuffle(unique_matches)

    # Still split matches roughly 80/20
    split_idx = int(len(unique_matches) * 0.8)
    train_matches = set(unique_matches[:split_idx])
    val_matches = set(unique_matches[split_idx:])

    # Count how many actual videos ended up in the validation group
    val_video_count = sum(1 for match in match_names if match in val_matches)

    # If the validation set has between 15% and 25% of total videos, break the loop!
    if total_videos * 0.15 <= val_video_count <= total_videos * 0.25:
        break

# Now build the final lists based on that perfect split
train_paths, train_labels = [], []
val_paths, val_labels = [], []

for path, label, match in zip(video_paths, labels, match_names):
    if match in train_matches:
        train_paths.append(path)
        train_labels.append(label)
    else:
        val_paths.append(path)
        val_labels.append(label)

print(f"📊 Training Videos: {len(train_paths)} | Validation Videos: {len(val_paths)}")

⚖️ Balancing the train/val split based on video counts...
📊 Training Videos: 4320 | Validation Videos: 1319


In [ ]:
class SoccerVideoDataset(Dataset):
    def __init__(self, video_paths, labels, is_train=True):
        self.video_paths = video_paths
        self.labels = labels
        self.is_train = is_train

        self.normalize = T.Normalize(
            mean=[0.43216, 0.394666, 0.37645],
            std=[0.22803, 0.22145, 0.216989]
        )

        self.color_jitter = T.ColorJitter(0.2, 0.2, 0.2)

        # 🔥 define once (better)
        self.weak_classes = [
            CLASS_TO_IDX["BALL_PLAYER_BLOCK"],
            CLASS_TO_IDX["CROSS"],
            CLASS_TO_IDX["SHOT"]
        ]

    def _load_video_frames(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)]

        frames = torch.tensor(np.array(frames), dtype=torch.float32) / 255.0
        frames = frames.permute(3, 0, 1, 2)
        return frames

    def _sample_exact_frames(self, frames):
        total_frames = frames.shape[1]

        if total_frames <= NUM_FRAMES:
            indices = torch.linspace(0, total_frames - 1, NUM_FRAMES).long()
        else:
            center = total_frames // 2
            half = NUM_FRAMES // 2
            start = center - half + random.randint(-2, 2)

            start = max(0, min(start, total_frames - NUM_FRAMES))
            indices = torch.arange(start, start + NUM_FRAMES)

        return frames[:, indices, :, :]

    def __getitem__(self, idx):
        frames = self._load_video_frames(self.video_paths[idx])
        label = self.labels[idx]

        # 1️⃣ sample frames first
        frames = self._sample_exact_frames(frames)

        # 2️⃣ augmentation
        if self.is_train:

            # 🔹 general augmentation (light)
            if random.random() > 0.5:
                frames = torch.flip(frames, dims=[3])

            # 🔥 stronger augmentation ONLY for weak classes
            if label in self.weak_classes:
                if random.random() > 0.5:
                    for t in range(frames.shape[1]):
                        frames[:, t] = self.color_jitter(frames[:, t])

        # 3️⃣ normalize LAST
        for t in range(frames.shape[1]):
            frames[:, t] = self.normalize(frames[:, t])

        return frames, label

    def __len__(self):
        return len(self.video_paths)

In [ ]:
model = models.r3d_18(weights=models.R3D_18_Weights.DEFAULT)

# Replace FC first
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),   # 🔥 reduced
    nn.Linear(num_features, len(CLASSES))
)

# Freeze early layers only
for name, param in model.named_parameters():
    if "layer3" in name or "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

model = model.to(device)

Downloading: "https://download.pytorch.org/models/r3d_18-b3b3357e.pth" to /root/.cache/torch/hub/checkpoints/r3d_18-b3b3357e.pth


100%|██████████| 127M/127M [00:01<00:00, 127MB/s]


In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = np.sqrt(class_weights)
class_weights = np.clip(class_weights, 1.0, 4.0)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,              # 🔥 better
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6         # 🔥 better
)

In [ ]:
from torch.utils.data import WeightedRandomSampler

# Create sample weights (one per video)
sample_weights = [class_weights[label]**0.7 for label in train_labels]

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    SoccerVideoDataset(train_paths, train_labels, is_train=True),
    batch_size=BATCH_SIZE,
    sampler=sampler   # 🔥 HERE
)

val_loader = DataLoader(
    SoccerVideoDataset(val_paths, val_labels, is_train=False),
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
print("🔥 Starting Training...")
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    scheduler.step() # Update learning rate

    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.2f}%")

    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_soccer_r3d_model.pth")
        print("   🌟 New best model saved!")

🔥 Starting Training...
Epoch [1/20] Train Loss: 1.4498 | Train Acc: 56.46% | Val Loss: 1.1265 | Val Acc: 62.85%
   🌟 New best model saved!
Epoch [2/20] Train Loss: 0.9171 | Train Acc: 72.31% | Val Loss: 1.0283 | Val Acc: 65.66%
   🌟 New best model saved!
Epoch [3/20] Train Loss: 0.6569 | Train Acc: 79.51% | Val Loss: 1.0310 | Val Acc: 69.22%
   🌟 New best model saved!
Epoch [4/20] Train Loss: 0.4258 | Train Acc: 87.31% | Val Loss: 1.0925 | Val Acc: 66.03%
Epoch [5/20] Train Loss: 0.3246 | Train Acc: 89.40% | Val Loss: 1.1097 | Val Acc: 66.41%
Epoch [6/20] Train Loss: 0.2514 | Train Acc: 92.55% | Val Loss: 1.1403 | Val Acc: 68.16%
Epoch [7/20] Train Loss: 0.2006 | Train Acc: 93.98% | Val Loss: 1.2628 | Val Acc: 67.17%
Epoch [8/20] Train Loss: 0.1534 | Train Acc: 95.88% | Val Loss: 1.1963 | Val Acc: 68.08%
Epoch [9/20] Train Loss: 0.1248 | Train Acc: 96.60% | Val Loss: 1.3155 | Val Acc: 67.55%
Epoch [10/20] Train Loss: 0.1027 | Train Acc: 97.04% | Val Loss: 1.2702 | Val Acc: 66.41%


KeyboardInterrupt: 

In [ ]:
!pip install ultralytics


In [ ]:
import torch
import cv2
import numpy as np
import torchvision.transforms as T
import torchvision.models.video as models
import torch.nn as nn
from ultralytics import YOLO
from collections import deque

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
IMAGE_SIZE = 112
NUM_FRAMES = 16
MODEL_PATH = "best_soccer_r3d_model.pth"

# ⚠️ YOUR INPUT AND OUTPUT VIDEO PATHS
INPUT_VIDEO_PATH = "/content/7.mp4"
OUTPUT_VIDEO_PATH = "/content/output_action2_tracking.mp4"

CLASS_NAMES = ['BALL_PLAYER_BLOCK', 'CROSS', 'HEADER', 'HIGH_PASS',
               'OUT', 'PASS', 'PLAYER_SUCCESSFUL_TACKLE', 'SHOT', 'THROW_IN']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. LOAD BOTH AI MODELS
# ==========================================
print("👀 Loading YOLO26 (The Player & Ball Tracker)...")
yolo_model = YOLO("yolo26n.pt") # YOLO26 Nano

print("🧠 Loading ResNet3D (The Action Analyzer)...")
r3d_model = models.r3d_18(weights=None)
r3d_model.fc = nn.Linear(r3d_model.fc.in_features, len(CLASS_NAMES))
r3d_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
r3d_model = r3d_model.to(device)
r3d_model.eval()

# ==========================================
# 3. REVISED HELPER: FIND PLAYER WITH THE BALL
# ==========================================
def get_active_player_crop(frame, last_box):
    """Finds the ball, finds the closest player's FEET, and returns crop."""
    # conf=0.15 forces YOLO to be less picky and find blurry soccer balls
    results = yolo_model(frame, classes=[0, 32], conf=0.15, verbose=False)[0]

    ball_center = None
    players = []

    for box in results.boxes:
        cls_id = int(box.cls[0].item())
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

        if cls_id == 32: # It's a ball
            # Only grab the first (highest confidence) ball to avoid stadium noise
            if ball_center is None:
                ball_center = (cx, cy)
        elif cls_id == 0: # It's a player
            # Save the center of their FEET (cx, y2), not their chest!
            players.append({"box": (int(x1), int(y1), int(x2), int(y2)), "feet": (cx, y2)})

    active_player_box = last_box # Default to keeping the SAME player if ball is lost

    # If we see the ball and players, calculate distance to FEET
    if ball_center is not None and len(players) > 0:
        min_dist = float('inf')
        for p in players:
            # Distance from ball to feet
            dist = np.sqrt((p["feet"][0] - ball_center[0])**2 + (p["feet"][1] - ball_center[1])**2)
            if dist < min_dist:
                min_dist = dist
                active_player_box = p["box"]

    # If it is the very first frame and we can't find the ball, pick the center-most player
    if active_player_box is None and len(players) > 0:
        h, w, _ = frame.shape
        screen_center = (w/2, h/2)
        active_player_box = min(players, key=lambda p: np.sqrt((p["feet"][0] - screen_center[0])**2 + (p["feet"][1] - screen_center[1])**2))["box"]

    # Failsafe
    if active_player_box is None:
        h, w, _ = frame.shape
        active_player_box = (w//4, h//4, w - w//4, h - h//4)

    # Crop the active player out of the frame
    px1, py1, px2, py2 = active_player_box
    pad = 30
    h, w, _ = frame.shape
    cy1, cy2 = max(0, py1 - pad), min(h, py2 + pad)
    cx1, cx2 = max(0, px1 - pad), min(w, px2 + pad)

    crop = frame[cy1:cy2, cx1:cx2]
    resized_crop = cv2.resize(crop, (IMAGE_SIZE, IMAGE_SIZE))

    return resized_crop, active_player_box

# ==========================================
# 4. PROCESS VIDEO & RENDER OUTPUT
# ==========================================
print(f"\n🎬 Processing and rendering video: {INPUT_VIDEO_PATH}")

cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Setup Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (width, height))

normalize = T.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])

# Rolling buffers to hold the last 16 frames of the active player
crop_buffer = deque(maxlen=NUM_FRAMES)
current_action = "Gathering Data..."
last_player_box = None
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_count += 1

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # 1. Track the active player and crop them
    player_crop, last_player_box = get_active_player_crop(rgb_frame, last_player_box)
    crop_buffer.append(player_crop)

    # 2. Every 5 frames (to save time), run ResNet3D if buffer is full
    if len(crop_buffer) == NUM_FRAMES and frame_count % 5 == 0:
        chunk = np.array(crop_buffer)
        chunk_tensor = torch.tensor(chunk, dtype=torch.float32) / 255.0
        chunk_tensor = chunk_tensor.permute(3, 0, 1, 2) # (C, T, H, W)

        for t in range(NUM_FRAMES):
            chunk_tensor[:, t, :, :] = normalize(chunk_tensor[:, t, :, :])

        input_tensor = chunk_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            outputs = r3d_model(input_tensor)
            probs = torch.softmax(outputs, dim=1)
            conf, pred_idx = torch.max(probs, dim=1)
            conf_score = conf.item() * 100

            # THE UPDATED NO ACTION LOGIC
            if conf_score > 60.0:
                current_action = f"{CLASS_NAMES[pred_idx.item()]} ({conf_score:.0f}%)"
            else:
                current_action = "NO ACTION"

    # 3. Draw on the frame!
    x1, y1, x2, y2 = last_player_box

    # Draw bounding box around the player
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Draw background box for text (makes it easier to read)
    text_size = cv2.getTextSize(current_action, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)[0]
    cv2.rectangle(frame, (x1, y1 - 30), (x1 + text_size[0], y1), (0, 255, 0), -1)

    # Put Action Text right above the player's head
    cv2.putText(frame, current_action, (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

    # Write to final video
    out.write(frame)

    if frame_count % 50 == 0:
        print(f"✅ Processed {frame_count} frames...")

cap.release()
out.release()
print(f"\n🎥 DONE! Your generated video is saved at: {OUTPUT_VIDEO_PATH}")